# 🧬 Genetic Variant Detection — Lab Notebook

**Sample:** NA12878 (HG001) — chromosome 20, region 31.1–32.1 Mb  
**Toolchain:** BWA-MEM · samtools · bcftools (no Java, no GATK)  


## Overview

This lab walks through the core pipeline for detecting germline single-nucleotide variants (SNVs) and small insertions/deletions (indels) from short-read sequencing data. Using a small, publicly available dataset.

**You will:**
1. Work on a human reference genome (chr20 only, hg19) and matched sequencing reads from public repositories
2. Align reads with BWA-MEM and assess mapping quality (MAPQ) to filter low-confidence alignments
3. Call SNVs and indels with `bcftools mpileup` + `bcftools call`
4. Apply hard filters and understand the difference between QUAL, GQ, and DP
5. **Distinguish true variants from strand bias artifacts, base quality errors, and repeat-region noise**
6. Compare your callset against the GIAB truth set and the 1000 Genomes panel


> **Why bcftools instead of GATK?** It is a single lightweight binary with no Java dependency, no read-group or sequence-dictionary requirements, and it runs in seconds on a 1 Mb region. It teaches the same concepts. The trade-off: it is pileup-based, so it is weaker on indels than HaplotypeCaller's local reassembly — negligible for an SNV-focused demo.

> **Why chr20 only from an older genome assembly** Managable file size; same concepts. 

> **Note:** This notebook uses Bash cells. If using a standard Python kernel, prefix shell commands with `!` or run them in a terminal alongside.


---
## Data Sources

All files are freely available — no login required. Using **chromosome 20** only keeps downloads and compute well under one hour.

| Data | Source | Notes | Size |
|---|---|---|---|
| Reference genome (chr20, GRCh38) | UCSC Genome Browser | Direct download | ~65 MB |
| Sequencing reads | 1000 Genomes Project (NA12878) | Stream a 1 Mb slice via samtools | ~50 MB |
| High-confidence variants (truth set) | Genome in a Bottle / NIST | Per-sample confident-region BED + VCF | ~450 MB compressed |
| Population variant panel | 1000 Genomes phase 3 | Stream or download chr20 subset | ~2 GB compressed |

**Why NA12878?** The most widely benchmarked human sample in existence. GIAB has independently validated thousands of variants in it, giving a gold-standard truth set to evaluate your calls against.

**Why chr20?** The smallest autosome (~64 Mb). Alignment and variant calling finish in minutes while covering repetitive regions, gene-dense areas, and known difficult loci.

**Why compare with GIAB and 1000G?** The 1000G panel is excellent for allele frequency lookups, and GIAB provides per-sample confidence intervals — flagging regions where calls are reliable — which is exactly what you need to distinguish true positives from false positives.

### Step 00: Tools and Conda Environment Required for This Lab.

Tools needed for this lab: fastqc, samtools, bcftools and bwa (for reads mapping)

| Tool | Version | Purpose |
|---|---|---|
| BWA-MEM | ≥ 0.7.17 | Align reads to reference |
| samtools | ≥ 1.17 | Sort, index, mark duplicates, depth, flagstat |
| bcftools | ≥ 1.17 | Call variants, filter, annotate, compare VCFs |
| FastQC | any | Read quality check |

These tools are available through the **miniconda** management system and the **bio2** environment. 

Conda: /nobackup/yye/Bio2/miniconda

Environment: bio2 (/nobackup/yye/Bio2/miniconda/envs/bio2)

** Terminal/Command line **

init + activate

```bash
conda init /nobackup/yye/Bio2/miniconda/bin/conda
```
If conda init ... doesn't work, try the following,
```bash
/nobackup/yye/Bio2/miniconda/bin/conda init bash
```
When this step is done, you can activate assembly as following,
```bash
conda activate bio2
```

** In VS Code (jupyter notebook) **
1. Remote-connect to silo via VS Code.
2. Clone this notebook to your personal folder on silo.
3. Open the Command Palette → **Python: Select Interpreter** and set  
   `/nobackup/yye/Bio2/miniconda/envs/bio2/bin/python` as the interpreter.

Optional: you may configure VS Code to automatically set the environment:
* In your project root folder, open (or create) .vscode/settings.json file.
* Add the python.defaultInterpreterPath setting pointing to shared path:

"python.defaultInterpreterPath": "/nobackup/yye/Bio2/miniconda/envs/bio2/bin/python"


## Step 1 — Setup (Data and Making Index File)

### 1a. Create a working folder for this lab


In [ ]:
mkdir -p lab08
cd lab08


### 1b. Copy these files to your working directory:

In [ ]:
cp /nobackup/yye/Bio2/datasets/variant/chr20.fa .
cp /nobackup/yye/Bio2/datasets/variant/reads_R1.fastq.gz .
cp /nobackup/yye/Bio2/datasets/variant/reads_R2.fastq.gz .

### 1c. Make index files for reads mapping

In [ ]:
bwa index chr20.fa
samtools faidx chr20.fa

### 1d. Check read quality with FastQC

**What to look for:**
- Per-base quality should stay above **Q20** for most of the read length. A sharp drop at the 3′ end is normal for Illumina.
- Adapter content warnings are fine here — BWA will soft-clip adapters during alignment.
- GC content should be roughly normal, centred near **41%** (human).
- If you see bimodal GC content or duplication rates >50%, the region may be repeat-heavy — note this for Step 4.

In [ ]:
fastqc reads_R1.fastq.gz reads_R2.fastq.gz -o ~./results/
# Open the generated HTML report in your browser to review

## Step 2 — Read Alignment and Mapping Quality (~10 min)

### 2a. Align with BWA-MEM, output results under ./results folder

The `-R` read group tag is optional for bcftools, but it is good practice: it labels the sample in the VCF header so multi-sample merges work later.

In [ ]:

bwa mem -t 4 -R '@RG\tID:NA12878\tSM:NA12878\tPL:ILLUMINA' chr20.fa reads_R1.fastq.gz reads_R2.fastq.gz | samtools sort -@ 4 -o results/NA12878_chr20.bam

samtools index results/NA12878_chr20.bam
samtools flagstat results/NA12878_chr20.bam

### 2b. Assess mapping quality — a critical filter ⚠️

Mapping quality (MAPQ) is a **Phred-scaled probability that a read is mapped to the wrong location**.
- MAPQ 20 → 1% chance of wrong placement
- MAPQ 60 → essentially zero (BWA-MEM's maximum for uniquely mapping reads)

| MAPQ | Interpretation | Action |
|---|---|---|
| 60 | Uniquely mapped, high confidence | Include |
| 20–59 | Likely correct, some ambiguity | Include with caution |
| 1–19 | Multi-mapping or near-repeat | Exclude from variant calls |
| 0 | Maps equally well to ≥ 2 locations | **Exclude** — causes false calls in repeat regions |

Unlike GATK, **bcftools does not filter on MAPQ by default** (`-q 0`). In Step 3 you will set `-q 20` explicitly — a useful teaching moment, because the filter is visible rather than hidden.

In [ ]:
# Distribution of MAPQ scores — what does your data look like?
samtools view -F 4 results/NA12878_chr20.bam | awk '{print $5}' | sort -n | uniq -c | sort -rn | head -20

# Fraction mapping at MAPQ >= 20
MAPQ20=$(samtools view -c -q 20 results/NA12878_chr20.bam)
TOTAL=$(samtools view -c -F 4 results/NA12878_chr20.bam)
echo "High-quality mapped reads (MAPQ>=20): $MAPQ20 / $TOTAL"
echo "Fraction: $(echo "scale=3; $MAPQ20 / $TOTAL" | bc)"

### 2c. Mark PCR duplicates

PCR duplicates are identical read pairs from the same original DNA molecule. Counting them as independent evidence **inflates depth** and creates false high-confidence variant calls — especially at heterozygous sites, where duplicates carry only one allele.

`samtools markdup` needs mate information (`fixmate -m`) and a coordinate-sorted BAM, hence the three-step pipeline.

> **Checkpoint:** If duplication rate > 50%, the library is likely over-amplified. Calls will still work but heterozygous sites may appear artificially homozygous.

In [ ]:
# fixmate requires name-sorted input; markdup requires coordinate-sorted input
samtools sort -n -@ 4 results/NA12878_chr20.bam \
  | samtools fixmate -m - - \
  | samtools sort -@ 4 - \
  | samtools markdup -s - results/NA12878_chr20_markdup.bam

samtools index results/NA12878_chr20_markdup.bam

# Duplication rate: the -s flag printed stats to stderr above.
# Cross-check with flagstat (look at the "duplicates" line):
samtools flagstat results/NA12878_chr20_markdup.bam | grep -i duplicate


## Step 3 — Variant Calling with bcftools

`bcftools mpileup` summarises the bases observed at every position (a **pileup**); `bcftools call` applies a genotyping model to decide which positions carry a variant.

### 3a. Pileup and call in one pipe

Key options:
- `-q 20` → ignore reads with MAPQ < 20 (the mapping-quality filter from Step 2b)
- `-Q 20` → ignore bases with base quality < 20
- `-a AD,DP,SP` → add per-sample allele depth, depth, and strand-bias P-value to FORMAT
- `call -m` → multiallelic caller (the modern default); `-v` → output variant sites only

In [ ]:
bcftools mpileup -f chr20.fa -q 20 -Q 20 -a AD,DP,SP \
  results/NA12878_chr20_markdup.bam \
  | bcftools call -mv -Oz -o results/NA12878_chr20_raw.vcf.gz

bcftools index results/NA12878_chr20_raw.vcf.gz

# How many raw variants?
bcftools stats results/NA12878_chr20_raw.vcf.gz | grep ^SN

### 3b. Understand the VCF fields — the most important columns

| VCF field | What it measures | Good threshold |
|---|---|---|
| QUAL | Phred-scaled probability the variant is real | ≥ 30 for reliable calls |
| GQ (FORMAT) | Genotype quality — confidence in HOM/HET assignment | ≥ 20 |
| DP (INFO / FORMAT) | Read depth at this site | ≥ 10; > 3× mean may indicate repeat collapse |
| AD (FORMAT) | Allele depth: reads supporting ref, alt | ~50/50 for HET; ~0/all for HOM-ALT |
| MQ (INFO) | Root-mean-square mapping quality at this site | ≥ 40 |
| SP (FORMAT) | Phred-scaled strand-bias P-value | < 30 |
| RPBZ (INFO) | Read-position bias Z-score (alt alleles clustered at read ends?) | between −3 and 3 |
| MQBZ (INFO) | Mapping-quality bias Z-score (alt reads lower MAPQ than ref reads?) | > −3 |
| SCBZ (INFO) | Soft-clip bias Z-score (alt reads more soft-clipped?) | < 3 |


⚠️ **QUAL alone is not enough.** A site can have QUAL=500 but a huge `SP` value — meaning the alt allele is only seen on one strand. That is a classic sequencing error signature.

In [ ]:
# View the header lines describing each INFO and FORMAT tag
bcftools view -h results/NA12878_chr20_raw.vcf.gz | grep -E '^##(INFO|FORMAT)'

# View the first 5 variant records
bcftools view -H results/NA12878_chr20_raw.vcf.gz | head -5


## Step 4 — Distinguishing True Variants from Errors

**This could be the most challenging part of the lab (and any variant detection practice).** Raw variant callers are sensitive by design — they flag anything that *could* be real. Your job is to filter confidently.

### The five main sources of false variants

**1. Strand bias (SP / SCBZ)**  
True variants appear on both strands at roughly equal frequency. If an alt allele is seen **only on forward reads**, it is almost certainly a sequencing or library preparation error. Verify visually in IGV by looking at read strand coloring.

**2. Low mapping quality (MQ / MQBZ)**  
If reads supporting a variant have low MAPQ (< 40), or markedly lower MAPQ than the ref-supporting reads (MQBZ strongly negative), they may be mismapped from elsewhere in the genome — common in segmental duplications. The variant is real *somewhere*; just not here.

**3. Repeat-region noise (RPBZ)**  
Variants consistently appearing near read ends (not randomly distributed across read positions) are likely alignment artifacts. RPBZ measures this — values far from zero flag these sites.

**4. Low quality by depth**  
bcftools has no QD tag, but the idea is the same: a QUAL of 30 at 2× depth is very uncertain; a QUAL of 30 at 30× depth is solid. Compute QUAL/DP yourself and treat < 2 as weak evidence.

**5. Depth outliers (DP)**  
Sites with extremely high depth (> 3× mean) often sit in collapsed repeats — the aligner stacks reads from multiple genome locations onto one. These create convincing-looking but false heterozygous calls.

---
### 4a. Apply hard filters

`bcftools filter -s NAME -e EXPR` tags records matching the expression with NAME in the FILTER column (soft filter) rather than removing them, so you can inspect what failed and why. Filter thresholds are **not** separated for SNPs and indels here; see the one-liners at the end for how to do that.

In [ ]:
# First, find the mean depth so you can set the high-DP cutoff
MEANDP=$(samtools depth -a results/NA12878_chr20_markdup.bam \
  | awk '{sum+=$3; n++} END {printf "%.0f", sum/n}')
MAXDP=$((MEANDP * 3))
echo "Mean depth: $MEANDP   High-depth cutoff (3x): $MAXDP"

In [ ]:
# Soft-filter: each failing site is tagged with the reason
bcftools filter -s LowQual    -e 'QUAL < 30'                 results/NA12878_chr20_raw.vcf.gz \
  | bcftools filter -s LowDP   -e "INFO/DP < 10" \
  | bcftools filter -s HighDP  -e "INFO/DP > $MAXDP" \
  | bcftools filter -s LowMQ   -e 'INFO/MQ < 40' \
  | bcftools filter -s LowQD   -e 'QUAL/INFO/DP < 2' \
  | bcftools filter -s StrandBias -e 'FMT/SP > 30 || INFO/SCBZ > 3' \
  | bcftools filter -s ReadPos -e 'INFO/RPBZ < -3 || INFO/RPBZ > 3' \
  | bcftools filter -s MQBias  -e 'INFO/MQBZ < -3' \
  -Oz -o results/filtered.vcf.gz
bcftools index results/filtered.vcf.gz

# Which filters fired, and how often?
bcftools query -f '%FILTER\n' results/filtered.vcf.gz | sort | uniq -c | sort -rn

In [ ]:
# Keep only PASS records as the final callset
bcftools view -f PASS results/filtered.vcf.gz -Oz -o results/final_calls.vcf.gz
bcftools index results/final_calls.vcf.gz

echo "Raw variants:        $(bcftools view -H results/NA12878_chr20_raw.vcf.gz | wc -l)"
echo "Final PASS variants: $(bcftools view -H results/final_calls.vcf.gz | wc -l)"

### 4b. Look at what failed — and why

Pull up the strand-bias failures with their supporting evidence. Pick one or two positions to inspect in IGV.

In [ ]:
# Sites that failed StrandBias: position, QUAL, depth, allele depth, strand-bias score
bcftools query -i 'FILTER~"StrandBias"' \
  -f '%CHROM\t%POS\t%REF\t%ALT\tQUAL=%QUAL\tDP=%INFO/DP\tAD=[%AD]\tSP=[%SP]\tSCBZ=%INFO/SCBZ\n' \
  results/filtered.vcf.gz | head -10

# Highest-depth sites — likely collapsed repeats
bcftools query -f '%INFO/DP\t%CHROM\t%POS\t%FILTER\n' results/filtered.vcf.gz \
  | sort -k1 -rn | head -10

## Step 5 — Annotation and Comparison to Reference Panels (optional)

### 5a. Download the GIAB truth set for NA12878

In [ ]:
# GIAB benchmark VCF, index, and confident-region BED — GRCh37 release
BASE="https://ftp-trace.ncbi.nlm.nih.gov/ReferenceSamples/giab/release/NA12878_HG001/latest/GRCh37"
wget ${BASE}/HG001_GRCh37_1_22_v4.2.1_benchmark.vcf.gz
wget ${BASE}/HG001_GRCh37_1_22_v4.2.1_benchmark.vcf.gz.tbi
wget ${BASE}/HG001_GRCh37_1_22_v4.2.1_benchmark.bed

# Confirm the contig naming matches ours (expect "20", not "chr20")
bcftools view -H HG001_GRCh37_1_22_v4.2.1_benchmark.vcf.gz -r 20:31100000-31200000 | head -2 | cut -f1-5

# Restrict to your 1 Mb region
bcftools view -r 20:31100000-32100000 \
  HG001_GRCh37_1_22_v4.2.1_benchmark.vcf.gz \
  -Oz -o giab_chr20_region.vcf.gz
bcftools index giab_chr20_region.vcf.gz

### 5b. Compute precision and recall

`bcftools isec` creates 4 output files:
- `0000.vcf` → private to your calls = **false positives**
- `0001.vcf` → private to GIAB = **false negatives** (variants you missed)
- `0002.vcf` → in both = **true positives** (from your callset)
- `0003.vcf` → in both = **true positives** (from GIAB)

Restricting to the GIAB confident-region BED (`-T`) is important: outside those regions GIAB makes no claim, so a call there is neither a TP nor an FP.

In [ ]:
bcftools isec -p results/isec_out \
  -T HG001_GRCh37_1_22_v4.2.1_benchmark.bed \
  results/final_calls.vcf.gz \
  giab_chr20_region.vcf.gz

TP=$(bcftools view -H results/isec_out/0002.vcf | wc -l)
FP=$(bcftools view -H results/isec_out/0000.vcf | wc -l)
FN=$(bcftools view -H results/isec_out/0001.vcf | wc -l)

echo "True positives:  $TP"
echo "False positives: $FP  (your calls not in GIAB)"
echo "False negatives: $FN  (GIAB variants you missed)"
echo "Precision: $(echo "scale=3; $TP / ($TP + $FP)" | bc)"
echo "Recall:    $(echo "scale=3; $TP / ($TP + $FN)" | bc)"

> **Interpreting your numbers:** A Precision of ~0.95 and Recall of ~0.80–0.85 is typical for low-coverage (4–6×) single-sample WGS with bcftools. Recall is limited mostly by depth: at 5× coverage many heterozygous sites simply have too few alt reads to call. High-coverage (30×) data typically achieves > 0.99 on both for SNPs.

### 5c. Annotate with 1000 Genomes allele frequencies

The 1000 Genomes phase 3 panel tells you whether a variant is known in the population and at what frequency. A variant **absent from 1000G** in your region may be rare, novel — or a false positive.

In [ ]:
# Download just the 1000G index (VCF streamed on the fly)
G1000="https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/ALL.chr20.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz"
wget ${G1000}.tbi

# Annotate your calls with 1000G ID (rsID) and allele frequency
bcftools annotate \
  -a ${G1000} \
  -c ID,INFO/AF \
  -r 20:31100000-32100000 \
  results/final_calls.vcf.gz \
  -Oz -o results/annotated_calls.vcf.gz

# View variants with their 1000G allele frequency
bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\t%ID\t%INFO/AF\n' \
  results/annotated_calls.vcf.gz | head -20

# How many are novel (not in 1000G)?
bcftools view -H results/annotated_calls.vcf.gz \
  | awk -F'\t' '$3=="." {count++} END {print count+0, "novel variants"}'

---
## ✅ Checkpoint Questions

Answer these using your actual output numbers. Fill in your answers below each question.

**Q1. Mapping quality.**  
What fraction of your reads mapped at MAPQ ≥ 20? What does a MAPQ 0 read physically represent, and why does including it in variant calling cause false positives? Try re-running Step 3a with `-q 0` — how does the raw variant count change?

**Answer:**

**Q2. Strand bias.**  
Look up one variant in `filtered.vcf.gz` tagged `StrandBias`. What are its `SP` and `SCBZ` values? Examine it in IGV — do the alt-supporting reads cluster on one strand?

**Answer:**

**Q3. Depth and duplication.**  
What was your duplication rate from `samtools markdup`? If you doubled the input read depth, would you expect precision to increase, decrease, or stay roughly the same — and why?

**Answer:**

**Q4. QUAL vs depth.**  
You have two variants: Site A has QUAL=200, DP=100; Site B has QUAL=50, DP=5. Calculate QUAL/DP for each. Which is more likely to be a true variant? What biological scenario would produce a high QUAL at very low depth?

**Answer:**

In [ ]:
# Calculate QUAL/DP for Site A and Site B
echo "Site A QUAL/DP: $(echo 'scale=2; 200/100' | bc)"
echo "Site B QUAL/DP: $(echo 'scale=2; 50/5' | bc)"

**Q5. Precision and recall.**  
Report your precision and recall from the GIAB comparison. If you relaxed the `SP` threshold from 30 to 60, which metric would change more — precision or recall? Why?

**My results:** Precision = `_______`, Recall = `_______`

**Answer:**

**Q6. 1000G cross-reference.**  
How many of your PASS calls were absent from the 1000G phase 3 panel? Of those, how many overlap a GIAB high-confidence region? What are three possible explanations for a variant that is PASS, in a GIAB confident region, but absent from 1000G?

**Answer:**

### Useful one-liners for quick exploration

In [ ]:
# Count PASS vs filtered, split by SNP / indel
bcftools view -v snps   results/filtered.vcf.gz | bcftools query -f '%FILTER\n' | sort | uniq -c | sed 's/^/SNP   /'
bcftools view -v indels results/filtered.vcf.gz | bcftools query -f '%FILTER\n' | sort | uniq -c | sed 's/^/INDEL /'

# Apply looser thresholds to indels only (strand/position bias is noisier for indels)
bcftools filter -i 'TYPE="indel" && QUAL>=30 && INFO/DP>=10' results/NA12878_chr20_raw.vcf.gz | bcftools view -H | wc -l

# Depth at a specific site
samtools depth -r chr20:31500000-31500001 results/NA12878_chr20_markdup.bam

# All PASS SNPs with genotype, GQ, DP and allele depths
bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\t[%GT\t%GQ\t%DP\t%AD]\n' \
  -i 'TYPE="snp"' results/final_calls.vcf.gz | head

# Genotype breakdown of your callset (HET vs HOM-ALT)
bcftools query -f '[%GT]\n' results/final_calls.vcf.gz | sort | uniq -c

---
## Further Reading

- [bcftools manual](https://samtools.github.io/bcftools/bcftools.html) — `mpileup`, `call`, `filter`, `isec`, `annotate`, `query`
- [bcftools filtering expressions](https://samtools.github.io/bcftools/howtos/filtering.html) — syntax for `-i` / `-e`
- [samtools markdup](https://www.htslib.org/doc/samtools-markdup.html) — the fixmate → markdup workflow
- [Genome in a Bottle (GIAB) data portal](https://www.nist.gov/programs-projects/genome-bottle) — benchmark VCFs and confident-region BEDs
- [1000 Genomes Project data portal](https://www.internationalgenome.org/data/) — phase 3 VCFs and sample metadata
- [GATK Best Practices](https://gatk.broadinstitute.org/hc/en-us/articles/360035535932) — for comparison, once students are comfortable with the bcftools pipeline